# MML Ch 6 — Probability and Distributions

**Source:** Mathematics for Machine Learning (Deisenroth, Faisal, Ong), Chapter 6  
**Notes:** `study/ml-math/notes/probability-distributions.md`

This notebook works through the key distributions, Bayes' theorem, the multivariate
Gaussian (marginals and conditionals), Cholesky sampling, the exponential family,
and the change-of-variables / inverse-CDF technique.

---

## Contents
1. PMF / PDF / CDF — Binomial and Gaussian
2. Bayes' Theorem — Disease-test numerical example
3. Multivariate Gaussian — 2D plot with correlation ellipses
4. Conditional Gaussian — analytical formula vs. samples
5. Cholesky Sampling — sample from N(μ, Σ) and verify covariance
6. Exponential Family — Gaussian and Binomial in canonical form
7. Change of Variables — transform U[0,1] → Exponential via inverse CDF

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import binom, norm, multivariate_normal

# Consistent style across all plots
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
rng = np.random.default_rng(42)

---
## 1. PMF / PDF / CDF

**Binomial PMF** — the probability of exactly k successes in n trials with success
probability p:
$$P(X=k) = \binom{n}{k} p^k (1-p)^{n-k}$$

**Gaussian PDF** — the bell curve:
$$p(x \mid \mu, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

**CDF** — cumulative: $F_X(x) = P(X \le x) = \int_{-\infty}^{x} p(t)\,dt$

> **Key distinction:** the PDF value p(x) is a *density*, not a probability
> — it can exceed 1. Only intervals have nonzero probability.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# --- Binomial PMF ---
ax = axes[0]
n, p_binom = 20, 0.35
k = np.arange(0, n + 1)
pmf = binom.pmf(k, n, p_binom)
ax.bar(k, pmf, color='steelblue', alpha=0.8, width=0.6)
ax.axvline(n * p_binom, color='crimson', ls='--', lw=1.5, label=f'mean = {n*p_binom:.1f}')
ax.set_title(f'Binomial PMF  (n={n}, p={p_binom})')
ax.set_xlabel('k (successes)')
ax.set_ylabel('P(X = k)')
ax.legend(fontsize=10)

# --- Gaussian PDF ---
ax = axes[1]
x = np.linspace(-4, 4, 400)
for mu, sigma, color, lbl in [(0, 1, 'steelblue', 'N(0,1)'),
                               (1, 0.5, 'darkorange', 'N(1,0.25)'),
                               (-1, 1.5, 'seagreen', 'N(-1,2.25)')]:
    ax.plot(x, norm.pdf(x, mu, sigma), color=color, lw=2, label=lbl)
ax.set_title('Gaussian PDF')
ax.set_xlabel('x')
ax.set_ylabel('p(x)')
ax.legend(fontsize=10)

# --- Gaussian CDF ---
ax = axes[2]
for mu, sigma, color, lbl in [(0, 1, 'steelblue', 'N(0,1)'),
                               (1, 0.5, 'darkorange', 'N(1,0.25)'),
                               (-1, 1.5, 'seagreen', 'N(-1,2.25)')]:
    ax.plot(x, norm.cdf(x, mu, sigma), color=color, lw=2, label=lbl)
ax.axhline(0.5, color='gray', ls=':', lw=1)
ax.set_title('Gaussian CDF')
ax.set_xlabel('x')
ax.set_ylabel('F(x) = P(X ≤ x)')
ax.legend(fontsize=10)

fig.suptitle('Section 6.2 — PMF, PDF, CDF', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. Bayes' Theorem — Disease Test

**Scenario:** A disease has prevalence 0.1% (rare). A diagnostic test has:
- Sensitivity (true positive rate): 99%  
- Specificity (true negative rate): 95%  

**Question:** Given a *positive test result*, what is the probability of having the disease?

$$P(D^+ \mid T^+) = \frac{P(T^+ \mid D^+)\, P(D^+)}{P(T^+)}$$

where $P(T^+) = P(T^+ \mid D^+)P(D^+) + P(T^+ \mid D^-)P(D^-)$.

In [ ]:
# --- Parameters ---
sensitivity  = 0.99   # P(T+ | D+)
specificity  = 0.95   # P(T- | D-)
prevalence   = 0.001  # P(D+)

p_disease    = prevalence
p_no_disease = 1 - prevalence
p_pos_given_disease    = sensitivity              # true positive
p_pos_given_no_disease = 1 - specificity          # false positive

# Total probability of a positive test (law of total probability)
p_positive = (p_pos_given_disease * p_disease +
              p_pos_given_no_disease * p_no_disease)

# Bayes' theorem
posterior = (p_pos_given_disease * p_disease) / p_positive

print("=== Disease-Test: Bayes' Theorem ===")
print(f"  P(D+)              = {p_disease:.4f}  (prevalence)")
print(f"  P(T+ | D+)         = {p_pos_given_disease:.4f}  (sensitivity)")
print(f"  P(T+ | D-)         = {p_pos_given_no_disease:.4f}  (false-positive rate)")
print(f"  P(T+) [total prob] = {p_positive:.4f}")
print()
print(f"  P(D+ | T+) = {posterior:.4f}  ({posterior*100:.2f}%)")
print()
print("Interpretation:")
print(f"  Despite a 99% sensitive test, a positive result carries only a ~{posterior*100:.1f}%")
print("  probability of disease, because the disease is so rare (base rate dominates).")

# --- Visual: how posterior changes with prevalence ---
prev_range = np.logspace(-4, 0, 300)  # 0.01% to 100%
posteriors = [
    (sensitivity * pr) / (sensitivity * pr + (1 - specificity) * (1 - pr))
    for pr in prev_range
]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(prev_range * 100, np.array(posteriors) * 100, 'steelblue', lw=2)
ax.axvline(prevalence * 100, color='crimson', ls='--', lw=1.5,
           label=f'prevalence = {prevalence*100:.1f}%')
ax.axhline(posterior * 100, color='darkorange', ls='--', lw=1.5,
           label=f'posterior  = {posterior*100:.2f}%')
ax.set_xscale('log')
ax.set_xlabel('Prevalence P(D+)  [%]  (log scale)')
ax.set_ylabel('P(D+ | T+)  [%]')
ax.set_title("Bayes' Theorem: Posterior vs. Prevalence\n"
             f"(sensitivity={sensitivity*100:.0f}%, specificity={specificity*100:.0f}%)")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

**Takeaway:** When base rates are low, even an accurate test produces many false
positives. This is the *base-rate fallacy* — the prior $P(D^+)$ dominates when
the disease is rare. Bayes' theorem makes this quantitative.

---
## 3. Multivariate Gaussian — 2D with Correlation Ellipses

For $\mathbf{X} \sim \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ in $\mathbb{R}^2$:

$$p(\mathbf{x}) = \frac{1}{2\pi|\boldsymbol{\Sigma}|^{1/2}}
  \exp\!\left(-\tfrac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1}(\mathbf{x}-\boldsymbol{\mu})\right)$$

The contours of constant density are **ellipses** whose axes align with the eigenvectors of $\boldsymbol{\Sigma}$.  
Here we use correlation $\rho = 0.8$:

$$\boldsymbol{\Sigma} = \begin{bmatrix} 1 & \rho \\ \rho & 1 \end{bmatrix}$$

In [ ]:
from matplotlib.patches import Ellipse

def covariance_ellipse(ax, mean, cov, n_std=1.0, **kwargs):
    """Draw a covariance ellipse at n_std standard deviations."""
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    # Chi-squared 2-DOF: 1σ ≈ 68%, 2σ ≈ 95%
    chi2_val = {1: 2.2977, 2: 5.9915}  # chi2.ppf(0.68,2) and chi2.ppf(0.95,2)
    scale = np.sqrt(chi2_val.get(n_std, n_std ** 2))
    width, height = 2 * scale * np.sqrt(vals)
    ellipse = Ellipse(xy=mean, width=width, height=height,
                      angle=angle, **kwargs)
    ax.add_patch(ellipse)

rho = 0.8
mu  = np.array([0.0, 0.0])
Sigma = np.array([[1.0, rho],
                  [rho, 1.0]])

# Grid for density
x1 = np.linspace(-3.5, 3.5, 200)
x2 = np.linspace(-3.5, 3.5, 200)
X1, X2 = np.meshgrid(x1, x2)
pos = np.stack([X1, X2], axis=-1)
Z = multivariate_normal.pdf(pos, mean=mu, cov=Sigma)

fig, ax = plt.subplots(figsize=(6, 5.5))
cf = ax.contourf(X1, X2, Z, levels=15, cmap='Blues', alpha=0.75)
plt.colorbar(cf, ax=ax, label='p(x₁, x₂)')

# 1σ and 2σ ellipses
covariance_ellipse(ax, mu, Sigma, n_std=1,
                   edgecolor='crimson', facecolor='none', lw=2.5, label='1σ ellipse')
covariance_ellipse(ax, mu, Sigma, n_std=2,
                   edgecolor='darkorange', facecolor='none', lw=2,
                   linestyle='--', label='2σ ellipse')
ax.plot(*mu, 'k+', ms=12, mew=2)

# Scatter samples
samples = rng.multivariate_normal(mu, Sigma, size=300)
ax.scatter(samples[:, 0], samples[:, 1], s=8, alpha=0.3, color='steelblue')

ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
ax.set_title(f'Bivariate Gaussian  (ρ = {rho})\n'
             r'$\mathcal{N}(\mathbf{0}, \Sigma)$ with 1σ and 2σ contours')
ax.set_aspect('equal')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Print eigendecomposition
eigvals, eigvecs = np.linalg.eigh(Sigma)
print(f"Eigenvalues of Σ: {eigvals}")
print(f"Eigenvectors:\n{eigvecs}")
print(f"\n→ Ellipse semi-axes lengths (1σ): {np.sqrt(eigvals * 2.2977)}")

---
## 4. Conditional Gaussian — Analytical vs. Samples

Given joint $(x_1, x_2) \sim \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$, the conditional
$p(x_1 \mid x_2 = c)$ is also Gaussian:

$$\mu_{1|2} = \mu_1 + \Sigma_{12}\Sigma_{22}^{-1}(c - \mu_2)$$
$$\sigma^2_{1|2} = \Sigma_{11} - \Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21}$$

This is the **Schur complement** of $\Sigma_{22}$ in $\Sigma$.

In [ ]:
# Use same (mu, Sigma) from the previous cell with rho=0.8
c = 1.0  # condition on x2 = 1

# Analytical conditional
mu1, mu2 = mu[0], mu[1]
S11, S12, S21, S22 = Sigma[0,0], Sigma[0,1], Sigma[1,0], Sigma[1,1]

mu_cond   = mu1 + S12 / S22 * (c - mu2)
var_cond  = S11 - S12 / S22 * S21
std_cond  = np.sqrt(var_cond)

print("=== Conditional Gaussian p(x₁ | x₂ = 1) ===")
print(f"  μ_{{1|2}}  = {mu_cond:.4f}  (analytical)")
print(f"  σ²_{{1|2}} = {var_cond:.4f}  (analytical — Schur complement)")

# Verify with samples: keep samples where x2 ≈ c (within a band)
band = 0.15
n_samples = 200_000
big_samples = rng.multivariate_normal(mu, Sigma, size=n_samples)
mask = np.abs(big_samples[:, 1] - c) < band
conditioned = big_samples[mask, 0]

print(f"\n  Sample mean  (x₂ ≈ {c}, band ±{band}) = {conditioned.mean():.4f}")
print(f"  Sample std                             = {conditioned.std():.4f}")
print(f"  Analytical std                         = {std_cond:.4f}")

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: joint density with conditioning line
ax = axes[0]
ax.contourf(X1, X2, Z, levels=12, cmap='Blues', alpha=0.7)
ax.axhline(c, color='crimson', lw=2, label=f'x₂ = {c} (condition)')
ax.axhline(c - band, color='crimson', lw=1, ls=':', alpha=0.6)
ax.axhline(c + band, color='crimson', lw=1, ls=':', alpha=0.6)
ax.scatter(conditioned[:500], np.full(min(500, len(conditioned)), c),
           s=4, alpha=0.3, color='darkorange', label='samples in band')
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_aspect('equal')
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
ax.set_title('Joint density with conditioning slice')
ax.legend(fontsize=9)

# Right: conditional distribution
ax = axes[1]
x_plot = np.linspace(-4, 4, 400)
ax.plot(x_plot, norm.pdf(x_plot, mu_cond, std_cond),
        color='crimson', lw=2.5, label=f'Analytical: N({mu_cond:.2f}, {var_cond:.2f})')
ax.hist(conditioned, bins=60, density=True, color='steelblue', alpha=0.5,
        label=f'Samples: mean={conditioned.mean():.2f}, std={conditioned.std():.2f}')
ax.axvline(mu_cond, color='crimson', ls='--', lw=1.5)
ax.set_xlabel('x₁')
ax.set_ylabel('density')
ax.set_title(f'p(x₁ | x₂ = {c})  — analytical vs samples')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 5. Cholesky Sampling

To draw $\mathbf{x} \sim \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ using the
Cholesky decomposition $\boldsymbol{\Sigma} = \mathbf{L}\mathbf{L}^\top$:

1. Sample $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$
2. Return $\mathbf{x} = \boldsymbol{\mu} + \mathbf{L}\mathbf{z}$

**Why it works:**
$\text{Cov}[\mathbf{L}\mathbf{z}] = \mathbf{L}\,\text{Cov}[\mathbf{z}]\,\mathbf{L}^\top
= \mathbf{L}\mathbf{I}\mathbf{L}^\top = \boldsymbol{\Sigma}$

In [ ]:
# Target distribution
mu_target   = np.array([2.0, -1.0])
Sigma_target = np.array([[4.0, 1.8],
                          [1.8, 1.0]])

# Cholesky decomposition
L = np.linalg.cholesky(Sigma_target)
print("=== Cholesky Decomposition ===")
print(f"L =\n{L}")
print(f"\nVerify L @ Lᵀ == Σ:\n{L @ L.T}")

# Sample using Cholesky
n_samples = 5000
z = rng.standard_normal((2, n_samples))  # z ~ N(0, I)
x_chol = mu_target[:, None] + L @ z       # x = mu + Lz

# Empirical statistics
sample_mean  = x_chol.mean(axis=1)
sample_cov   = np.cov(x_chol)

print(f"\n=== Comparison (n={n_samples} samples) ===")
print(f"Target μ:      {mu_target}")
print(f"Sample mean:   {sample_mean.round(3)}")
print(f"\nTarget Σ:\n{Sigma_target}")
print(f"Sample cov:\n{sample_cov.round(3)}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
ax.scatter(x_chol[0], x_chol[1], s=5, alpha=0.25, color='steelblue')
ax.plot(*mu_target, 'r+', ms=14, mew=2.5, label='True mean μ')
ax.plot(*sample_mean, 'ko', ms=7, label='Sample mean')
covariance_ellipse(ax, mu_target, Sigma_target, n_std=1,
                   edgecolor='crimson', facecolor='none', lw=2.5)
covariance_ellipse(ax, mu_target, Sigma_target, n_std=2,
                   edgecolor='darkorange', facecolor='none', lw=2, linestyle='--')
ax.set_aspect('equal')
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
ax.set_title(f'Cholesky samples from N(μ, Σ)\nn={n_samples}')
ax.legend(fontsize=9)

ax = axes[1]
labels = ['Σ₁₁', 'Σ₁₂', 'Σ₂₁', 'Σ₂₂']
true_vals   = [Sigma_target[0,0], Sigma_target[0,1], Sigma_target[1,0], Sigma_target[1,1]]
sample_vals = [sample_cov[0,0],   sample_cov[0,1],   sample_cov[1,0],   sample_cov[1,1]]
x_pos = np.arange(4)
ax.bar(x_pos - 0.2, true_vals,   0.35, label='True Σ',   color='steelblue', alpha=0.8)
ax.bar(x_pos + 0.2, sample_vals, 0.35, label='Sample cov', color='darkorange', alpha=0.8)
ax.set_xticks(x_pos); ax.set_xticklabels(labels)
ax.set_title('Target Σ vs. Sample Covariance')
ax.set_ylabel('value')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## 6. Exponential Family

A distribution belongs to the **exponential family** if:

$$p(x \mid \boldsymbol{\eta}) = h(x)\exp\!\bigl(\boldsymbol{\eta}^\top \boldsymbol{\phi}(x) - A(\boldsymbol{\eta})\bigr)$$

where $\boldsymbol{\eta}$ = natural parameters, $\boldsymbol{\phi}(x)$ = sufficient statistics,
$A(\boldsymbol{\eta})$ = log-partition function.

### 6a. Gaussian in canonical form

For $X \sim \mathcal{N}(\mu, \sigma^2)$ with known $\sigma^2$, treat $\mu$ as parameter:
$$h(x) = \frac{1}{\sqrt{2\pi\sigma^2}} e^{-x^2/(2\sigma^2)}, \quad
  \eta = \frac{\mu}{\sigma^2}, \quad
  \phi(x) = x, \quad
  A(\eta) = \frac{\mu^2}{2\sigma^2} = \frac{\eta^2\sigma^2}{2}$$

### 6b. Binomial in canonical form

For $X \sim \text{Bin}(n, p)$:
$$h(x) = \binom{n}{x}, \quad
  \eta = \ln\frac{p}{1-p}\text{ (log-odds)}, \quad
  \phi(x) = x, \quad
  A(\eta) = n\ln(1 + e^{\eta})$$

In [ ]:
# ---- Gaussian: verify that the exponential family form equals the standard form ----
mu_g, sigma2 = 2.0, 3.0
sigma = np.sqrt(sigma2)
eta_g = mu_g / sigma2          # natural parameter

x_vals = np.linspace(-4, 8, 300)

# Standard form
p_standard = norm.pdf(x_vals, mu_g, sigma)

# Exponential family form
A_eta = (eta_g ** 2 * sigma2) / 2  # log-partition function
h_x   = (1 / np.sqrt(2 * np.pi * sigma2)) * np.exp(-x_vals**2 / (2 * sigma2))
p_expfam = h_x * np.exp(eta_g * x_vals - A_eta)

print("=== Gaussian Exponential Family ===")
print(f"  μ = {mu_g},  σ² = {sigma2}")
print(f"  Natural parameter η = μ/σ² = {eta_g:.4f}")
print(f"  Log-partition A(η) = {A_eta:.4f}")
print(f"  Max |p_standard - p_expfam|: {np.max(np.abs(p_standard - p_expfam)):.2e}  (should be ~0)")

# ---- Binomial: verify exponential family form ----
n_bin, p_bin = 10, 0.4
eta_bin = np.log(p_bin / (1 - p_bin))   # log-odds
k_vals  = np.arange(0, n_bin + 1)

from scipy.special import comb
A_bin = n_bin * np.log(1 + np.exp(eta_bin))  # log(normalizing constant)
h_k   = comb(n_bin, k_vals, exact=False)     # binomial coefficients
p_expfam_bin = h_k * np.exp(eta_bin * k_vals - A_bin)
p_standard_bin = binom.pmf(k_vals, n_bin, p_bin)

print(f"\n=== Binomial Exponential Family ===")
print(f"  n = {n_bin},  p = {p_bin}")
print(f"  Natural parameter η = log(p/(1-p)) = {eta_bin:.4f}")
print(f"  Log-partition A(η) = {A_bin:.4f}")
print(f"  Max |p_standard - p_expfam|: {np.max(np.abs(p_standard_bin - p_expfam_bin)):.2e}")

# ---- Plot both ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(x_vals, p_standard, 'steelblue', lw=3, label='Standard N(μ,σ²)')
ax.plot(x_vals, p_expfam, 'crimson', lw=1.5, ls='--', label='Exp. family form')
ax.set_title('Gaussian: standard vs. exponential family')
ax.set_xlabel('x'); ax.set_ylabel('p(x)')
ax.legend(fontsize=10)
ax.text(0.05, 0.92,
        f'η={eta_bin:.3f}\nφ(x)=x\nA(η)={A_eta:.3f}',
        transform=ax.transAxes, fontsize=9,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax = axes[1]
w = 0.35
ax.bar(k_vals - w/2, p_standard_bin, w, label='Standard Bin(n,p)', color='steelblue', alpha=0.8)
ax.bar(k_vals + w/2, p_expfam_bin,   w, label='Exp. family form',  color='crimson',   alpha=0.7)
ax.set_title('Binomial: standard vs. exponential family')
ax.set_xlabel('k'); ax.set_ylabel('P(X = k)')
ax.legend(fontsize=10)
ax.text(0.62, 0.92,
        f'η={eta_bin:.3f}\nφ(k)=k\nA(η)={A_bin:.3f}',
        transform=ax.transAxes, fontsize=9,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

---
## 7. Change of Variables — Inverse CDF / Inverse Transform Sampling

**Theorem (§6.7):** If $U \sim \text{Uniform}[0,1]$ and $F_X$ is a CDF with inverse $F_X^{-1}$, then:

$$X = F_X^{-1}(U) \sim F_X$$

**Why it works:** $P(F_X^{-1}(U) \le x) = P(U \le F_X(x)) = F_X(x)$ ✓

**Example:** Transform $U[0,1]$ to $\text{Exponential}(\lambda)$.
- CDF: $F_X(x) = 1 - e^{-\lambda x}$
- Inverse: $F_X^{-1}(u) = -\frac{1}{\lambda}\ln(1-u)$

**Change-of-variables density check:**
$$p_X(x) = p_U(u)\left|\frac{du}{dx}\right| = 1 \cdot \lambda e^{-\lambda x} = \lambda e^{-\lambda x}$$

In [ ]:
lam = 1.5  # rate parameter
n_samples = 50_000

# Step 1: draw U ~ Uniform[0,1]
U = rng.uniform(0, 1, n_samples)

# Step 2: apply inverse CDF of Exponential(lambda)
X_samples = -np.log(1 - U) / lam

# Theoretical PDF
x_plot = np.linspace(0, 5, 400)
p_exponential = lam * np.exp(-lam * x_plot)

# Empirical statistics
true_mean = 1 / lam
true_var  = 1 / lam**2

print(f"=== Inverse CDF Sampling: Exponential(λ={lam}) ===")
print(f"  True mean:    {true_mean:.4f}")
print(f"  Sample mean:  {X_samples.mean():.4f}")
print(f"  True std:     {np.sqrt(true_var):.4f}")
print(f"  Sample std:   {X_samples.std():.4f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Left: uniform source
ax = axes[0]
ax.hist(U, bins=50, density=True, color='steelblue', alpha=0.8, edgecolor='white', lw=0.3)
ax.axhline(1.0, color='crimson', lw=2, ls='--', label='U[0,1] density = 1')
ax.set_xlabel('u'); ax.set_ylabel('density')
ax.set_title(f'Step 1: U ~ Uniform[0, 1]\n(n={n_samples:,})')
ax.set_ylim(0, 1.4)
ax.legend(fontsize=9)

# Middle: the transformation
ax = axes[1]
u_range = np.linspace(0.001, 0.999, 400)
x_transform = -np.log(1 - u_range) / lam
ax.plot(u_range, x_transform, 'steelblue', lw=2.5,
        label=r'$x = -\frac{1}{\lambda}\ln(1-u)$')
ax.set_xlabel('u  (uniform)')
ax.set_ylabel('x  (exponential)')
ax.set_title(f'Inverse CDF: F_X⁻¹(u)')
ax.legend(fontsize=10)

# Right: resulting exponential samples vs. theoretical PDF
ax = axes[2]
ax.hist(X_samples, bins=80, density=True, color='steelblue', alpha=0.7,
        edgecolor='white', lw=0.3, label='Samples')
ax.plot(x_plot, p_exponential, 'crimson', lw=2.5,
        label=f'Exponential(λ={lam}) PDF')
ax.set_xlabel('x')
ax.set_ylabel('density')
ax.set_title(f'Step 2: X = F⁻¹(U) ~ Exponential(λ={lam})')
ax.legend(fontsize=9)
ax.set_xlim(0, 5)

fig.suptitle('Section 6.7 — Change of Variables / Inverse Transform Sampling',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Summary

| Section | Key formula / insight |
|---------|----------------------|
| PMF / PDF / CDF | PDF is density (can exceed 1); CDF = integral of PDF |
| Bayes' theorem | $P(D^+|T^+) \propto P(T^+|D^+)P(D^+)$; rare events require large sensitivity |
| Multivariate Gaussian | Contours are ellipses aligned with eigenvectors of Σ |
| Conditional Gaussian | $\mu_{1|2} = \mu_1 + \Sigma_{12}\Sigma_{22}^{-1}(x_2-\mu_2)$; variance always reduces |
| Cholesky sampling | $\mathbf{x} = \boldsymbol{\mu} + L\mathbf{z}$, $z\sim\mathcal{N}(0,I)$, $\Sigma=LL^\top$ |
| Exponential family | $p(x|\eta) = h(x)\exp(\eta^\top\phi(x)-A(\eta))$; Gaussian and Binomial both fit |
| Change of variables | $X = F_X^{-1}(U)$ where $U\sim$ Uniform[0,1] yields $X\sim F_X$ |